# Day 7 — the direct token-to-logit share, computed exactly

The Day 6 figure (+0.556 with replacement, +0.033 with rating) was parsed from the **displayed**
trace: the logit's top four edges after thresholding, on the 40 rated graphs. That is a property of
what the viewer showed, not of the graph.

This notebook computes it from the **full influence vector** on every graph, with no thresholding
and no top-k, and re-runs the three claims that rest on it.

**The quantity.** Push influence backwards from the logits. It terminates at nodes with no inputs —
token embeddings and error nodes — and the embedding share is the replacement score. Split that
share by **path length**:

- `direct_share` — mass reaching an embedding in **one** step, straight from a logit node. The
  residual-stream path: the token's embedding read by the unembedding with no feature in between.
- `mediated_share` — mass reaching an embedding through **two or more** steps, so through at least
  one feature.

Both are fractions of total replacement, so `direct_share + mediated_share = 1`.

GPU needed. ~255 graphs at ~18 s is about 80 minutes; set `RATED_ONLY = True` for the 40 rated
graphs in ~12 minutes if you only want to firm up the Day 6 number.

## Part 0 — Install

In [1]:
!pip install -q circuit-tracer
print("installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fol

## Part 1 — Environment, inputs, model

In [8]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, time, gc, json, glob, shutil
import numpy as np, pandas as pd, torch
from scipy import stats

OUT = "/kaggle/working/out"; os.makedirs(OUT, exist_ok=True)

def find_input(name, required=True):
    hits = (glob.glob(f"/kaggle/working/**/{name}", recursive=True)
            + glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits and required:
        raise FileNotFoundError(name)
    return hits[0] if hits else None

def free():
    gc.collect(); torch.cuda.empty_cache()

src = find_input("ct_utils.py")
if os.path.abspath(src) != "/kaggle/working/ct_utils.py":
    shutil.copy(src, "/kaggle/working/ct_utils.py")
sys.path.insert(0, "/kaggle/working")
import ct_utils, importlib; importlib.reload(ct_utils)
from ct_utils import to_np

from kaggle_secrets import UserSecretsClient
_t = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
from huggingface_hub import login; login(token=_t)

corpus = pd.read_csv(find_input("corpus.csv"),
                     dtype={"expected": str, "subtype": str, "prompt": str}, keep_default_na=False)
CFG = json.load(open(find_input("day3_config.json")))
AK, NL = CFG["attr_kwargs"], CFG["n_layers"]
print(f"corpus {len(corpus)} prompts | cap {AK['max_feature_nodes']} | {NL} layers")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


corpus 255 prompts | cap 32768 | 26 layers


In [10]:
import sys, types
sys.modules["diffusers"] = types.ModuleType("diffusers")   # empty stub
from circuit_tracer import ReplacementModel, attribute

from circuit_tracer import ReplacementModel, attribute
model = ReplacementModel.from_pretrained("google/gemma-2-2b", "gemma", backend="nnsight",
                                         dtype=torch.bfloat16, device="cuda", lazy_encoder=True)
print("model loaded")

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

model loaded


## Part 2 — The measurement

Influence propagates backwards one edge per step, so the step at which mass lands on an embedding
**is** the path length. Accumulating the embedding mass per step costs nothing extra — it is the
same loop that already computes replacement score.

Three quantities per graph, all exact and all on the full matrix:

| | |
|---|---|
| `direct_mass` | embedding mass absorbed at step 1 |
| `total_emb_mass` | embedding mass absorbed at all steps = replacement score |
| `direct_share` | `direct_mass / total_emb_mass` |

A one-step path from a logit to an embedding exists only through the residual stream, since every
other route passes through at least one feature node.

In [11]:
@torch.no_grad()
def direct_path(gr, cfg, device="cuda"):
    '''Exact split of replacement score by path length. No thresholding, no top-k.'''
    A = gr.adjacency_matrix.to(device=device, dtype=torch.float32, copy=True).abs_()
    N = A.shape[0]
    nt = int(len(to_np(gr.input_tokens)))
    nlg = int(len(to_np(gr.logit_probabilities)))
    nl = cfg["n_layers"]
    n_feat = int(len(gr.selected_features))
    err_end, tok_end = n_feat + nl * nt, n_feat + nl * nt + nt
    if tok_end + nlg != N:
        raise ValueError(f"layout mismatch on {N} nodes")

    A.div_(A.sum(1, keepdim=True).clamp_min(1e-30))
    lw = torch.zeros(N, device=device)
    lw[N - nlg:] = torch.as_tensor(to_np(gr.logit_probabilities), device=device,
                                   dtype=torch.float32)
    lw /= lw.sum()

    emb = torch.arange(err_end, tok_end, device=device)
    err = torch.arange(n_feat, err_end, device=device)

    v, infl = lw.clone(), torch.zeros(N, device=device)
    by_step_emb, by_step_err = [], []
    for step in range(1, 501):
        v = v @ A
        infl += v
        by_step_emb.append(float(v[emb].sum()))
        by_step_err.append(float(v[err].sum()))
        if float(v.sum()) < 1e-12:
            break

    tok_inf = float(infl[emb].sum()); err_inf = float(infl[err].sum())
    per_token_direct = (lw @ A)[emb].cpu().numpy()      # step-1 mass, per token position
    del A, v, infl; free()

    e = np.array(by_step_emb)
    return dict(
        replacement_score=tok_inf / (tok_inf + err_inf),
        total_emb_mass=tok_inf, total_err_mass=err_inf,
        direct_mass=e[0], direct_share=float(e[0] / tok_inf) if tok_inf > 0 else np.nan,
        mediated_share=float(1 - e[0] / tok_inf) if tok_inf > 0 else np.nan,
        emb_mass_by_step=e.tolist(), err_mass_by_step=by_step_err,
        direct_from_final_token=float(per_token_direct[-1] / max(e[0], 1e-12)),
        n_steps=len(e), n_tok=nt, n_features=n_feat)

print("ready")

ready


## Part 3 — Run

Resumable: rerunning skips graphs already recorded. Set `RATED_ONLY = True` to do only the 40
rated graphs first; the full corpus is what the regression needs.

In [28]:
RATED_ONLY = False
DP = f"{OUT}/direct_path.jsonl"

todo = corpus.copy()
if RATED_ONLY:
    man = pd.read_csv(find_input("rating_manifest.csv"))
    todo = corpus[corpus.prompt_id.isin(set(man.prompt_id))]

done = ({json.loads(l)["prompt_id"] for l in open(DP) if l.strip()}
        if os.path.exists(DP) else set())
todo = todo[~todo.prompt_id.isin(done)]
print(f"{len(done)} done, {len(todo)} to go")

t0 = time.time()
for i, (_, r) in enumerate(todo.iterrows(), 1):
    try:
        g = attribute(prompt=r.prompt, model=model, **AK)
        rec = dict(prompt_id=r.prompt_id, category=r.category, status="ok",
                   **direct_path(g, CFG))
        del g
    except Exception as ex:
        rec = dict(prompt_id=r.prompt_id, category=r.category, status="error",
                   error=f"{type(ex).__name__}: {str(ex)[:200]}")
    free()
    with open(DP, "a") as f:
        f.write(json.dumps(rec, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
    if i % 10 == 0 or rec["status"] != "ok":
        el = time.time() - t0
        print(f"{i:3d}/{len(todo)}  {r.category:22s} elapsed {el/60:5.1f}m  "
              f"eta {el/i*(len(todo)-i)/60:5.1f}m" +
              ("" if rec["status"] == "ok" else f"   <-- {rec['status']}"))
print(f"\nfinished: {len(done) + len(todo)} graphs")

41 done, 214 to go
 10/214  factual_recall         elapsed   2.6m  eta  52.4m
 20/214  factual_recall         elapsed   5.3m  eta  51.9m
 30/214  factual_recall         elapsed   7.6m  eta  46.8m
 40/214  arithmetic             elapsed   9.1m  eta  39.7m
 50/214  arithmetic             elapsed  10.8m  eta  35.3m
 60/214  syntactic_agreement    elapsed  14.5m  eta  37.3m
 70/214  syntactic_agreement    elapsed  17.0m  eta  35.0m
 80/214  multi_hop              elapsed  19.2m  eta  32.1m
 90/214  multi_hop              elapsed  22.1m  eta  30.4m
100/214  multi_hop              elapsed  24.4m  eta  27.8m
110/214  code_completion        elapsed  27.3m  eta  25.8m
120/214  code_completion        elapsed  29.5m  eta  23.1m
130/214  code_completion        elapsed  34.4m  eta  22.2m
140/214  induction              elapsed  40.7m  eta  21.5m
160/214  entity_known_unknown   elapsed  48.2m  eta  16.3m
170/214  entity_known_unknown   elapsed  50.2m  eta  13.0m
180/214  obfuscated             elaps

## Part 4 — Did it reproduce the Day 3 scores?

The replacement score recomputed here should match `results.csv` to the re-attribution noise floor (median 9e-05, max 3.4e-04). A larger gap means this implementation diverges from the corpus run and nothing below is trustworthy.

In [29]:
dp = pd.DataFrame([json.loads(l) for l in open(DP) if l.strip()])
dp = dp[dp.status == "ok"].copy()
res = pd.read_csv(find_input("results.csv"), keep_default_na=False)
res["replacement_score"] = pd.to_numeric(res.replacement_score, errors="coerce")

chk = dp.merge(res[["prompt_id", "replacement_score"]], on="prompt_id", suffixes=("_new", "_day3"))
d = (chk.replacement_score_new - chk.replacement_score_day3).abs()
print(f"n = {len(chk)}   max |diff| = {d.max():.2e}   median = {d.median():.2e}")
print(f"within the 3.4e-04 re-attribution ceiling: {(d <= 3.4e-4).mean():.1%}")
assert d.max() < 2e-3, "recomputed replacement diverges from the corpus run - stop here"
print("\nreproduces the corpus run")

n = 255   max |diff| = 5.83e-04   median = 7.80e-05
within the 3.4e-04 re-attribution ceiling: 97.6%

reproduces the corpus run


## Part 5 — The exact direct share

In [30]:
print(f"direct share of replacement, overall: mean {dp.direct_share.mean():.3f}, "
      f"median {dp.direct_share.median():.3f}, range {dp.direct_share.min():.3f}-{dp.direct_share.max():.3f}")
print(f"\nDay 6 estimate from displayed edges was 0.372 on the rated 40.")
print()
print(dp.groupby("category").direct_share.agg(["mean", "std", "min", "max", "size"])
        .round(3).sort_values("mean", ascending=False).to_string())

rho_all, p_all = stats.spearmanr(dp.direct_share, dp.replacement_score)
print(f"\nSpearman(direct share, replacement) across all {len(dp)} graphs = {rho_all:+.3f} (p = {p_all:.2e})")
print(f"Pearson  = {dp.direct_share.corr(dp.replacement_score):+.3f}")

direct share of replacement, overall: mean 0.095, median 0.091, range 0.023-0.184

Day 6 estimate from displayed edges was 0.372 on the rated 40.

                       mean    std    min    max  size
category                                              
induction             0.127  0.036  0.071  0.184    24
factual_recall        0.124  0.024  0.057  0.166    40
multi_hop             0.096  0.016  0.071  0.127    32
arithmetic            0.093  0.008  0.080  0.109    28
syntactic_agreement   0.090  0.021  0.054  0.130    26
entity_known_unknown  0.090  0.004  0.082  0.097    32
code_completion       0.077  0.029  0.023  0.133    29
obfuscated            0.070  0.016  0.042  0.101    44

Spearman(direct share, replacement) across all 255 graphs = +0.124 (p = 4.85e-02)
Pearson  = +0.157


In [31]:
# how much of the between-category spread in replacement is the direct component?
def eta2(v, g):
    v = np.asarray(v, float); g = np.asarray(g); gm = v.mean()
    sst = ((v - gm) ** 2).sum()
    ssb = sum(len(v[g == k]) * (v[g == k].mean() - gm) ** 2 for k in np.unique(g))
    return float(ssb / sst) if sst > 0 else np.nan

cat = dp.category.values
print(f"eta2 of category on replacement score : {eta2(dp.replacement_score, cat):.4f}")
print(f"eta2 of category on direct share      : {eta2(dp.direct_share, cat):.4f}")
print(f"eta2 of category on MEDIATED mass     : "
      f"{eta2(dp.total_emb_mass * dp.mediated_share, cat):.4f}")
print()
print("mediated replacement = the part that passed through at least one feature:")
dp["mediated_replacement"] = dp.replacement_score * dp.mediated_share
print(dp.groupby("category").mediated_replacement.agg(["mean", "std"]).round(4)
        .sort_values("mean", ascending=False).to_string())
print("\nIf the category ordering changes between replacement and mediated replacement,")
print("the metric's ranking is partly an artefact of the direct component.")

eta2 of category on replacement score : 0.2423
eta2 of category on direct share      : 0.4672
eta2 of category on MEDIATED mass     : 0.2843

mediated replacement = the part that passed through at least one feature:
                        mean     std
category                            
code_completion       0.6787  0.0217
obfuscated            0.6761  0.0198
entity_known_unknown  0.6758  0.0066
syntactic_agreement   0.6745  0.0229
multi_hop             0.6580  0.0238
induction             0.6509  0.0299
factual_recall        0.6470  0.0295
arithmetic            0.6434  0.0149

If the category ordering changes between replacement and mediated replacement,
the metric's ranking is partly an artefact of the direct component.


## Part 6 — Re-run the three claims that rested on the parsed estimate

1. Correlation with human rating — the Day 6 figure was +0.033 on 40 graphs.
2. Whether the length effect survives when the direct component is removed.
3. Whether P2's category ordering survives.

In [32]:
# 1. against the human ratings
rat = pd.DataFrame([json.loads(l) for l in open(find_input("ratings_js.jsonl")) if l.strip()])
rat = rat.sort_values("presentation").drop_duplicates("prompt_id")
J = rat[["prompt_id", "rating"]].merge(dp, on="prompt_id")
print(f"n = {len(J)}")
for col in ["direct_share", "replacement_score", "mediated_replacement"]:
    rho, p = stats.spearmanr(J[col], J.rating)
    print(f"Spearman({col:22s}, rating) = {rho:+.3f}  (p = {p:.3f})")
print()
print(J.groupby("rating")[["direct_share", "replacement_score", "mediated_replacement"]]
       .mean().round(4).to_string())
print("\nIf mediated replacement tracks rating better than replacement does, the direct")
print("component is the part that was obscuring the relationship.")

n = 40
Spearman(direct_share          , rating) = -0.089  (p = 0.586)
Spearman(replacement_score     , rating) = +0.243  (p = 0.132)
Spearman(mediated_replacement  , rating) = +0.297  (p = 0.063)

        direct_share  replacement_score  mediated_replacement
rating                                                       
0             0.1033             0.7313                0.6557
1             0.0888             0.7285                0.6637
2             0.1099             0.7551                0.6720

If mediated replacement tracks rating better than replacement does, the direct
component is the part that was obscuring the relationship.


In [33]:
# 2. the matched short/long frames, on direct share and on mediated replacement
fr = dp[dp.category.isin(["factual_recall", "multi_hop"])].merge(
    corpus[["prompt_id", "subtype", "expected"]], on="prompt_id")
fr["frame"] = np.where(fr.subtype.str.contains("long"), "long",
               np.where(fr.subtype.str.contains("med"), "med", "short"))

for col in ["replacement_score", "direct_share", "mediated_replacement"]:
    piv = fr.pivot_table(index="expected", columns="frame", values=col, aggfunc="mean")
    pair = piv[["short", "long"]].dropna()
    d = pair["long"] - pair["short"]
    w = stats.wilcoxon(pair["long"], pair["short"])
    print(f"{col:22s} n={len(pair)}  mean diff {d.mean():+.4f}  "
          f"{(d > 0).sum()}/{len(d)} positive  p = {w.pvalue:.2e}")

print("\nper-token direct mass is a fraction of step-1 mass at the final token:")
print(fr.groupby("frame")[["n_tok", "direct_share", "direct_from_final_token"]]
        .mean().round(3).to_string())

replacement_score      n=27  mean diff +0.0323  27/27 positive  p = 1.49e-08
direct_share           n=27  mean diff -0.0091  11/27 positive  p = 5.21e-02
mediated_replacement   n=27  mean diff +0.0352  26/27 positive  p = 2.98e-08

per-token direct mass is a fraction of step-1 mass at the final token:
        n_tok  direct_share  direct_from_final_token
frame                                               
long   13.889         0.106                    0.274
med    10.000         0.131                    0.464
short   9.030         0.108                    0.221


In [34]:
fr = dp[dp.category.isin(["factual_recall","multi_hop"])].merge(
        corpus[["prompt_id","subtype","expected"]], on="prompt_id")
print(fr.columns.tolist())
print(fr[["prompt_id","subtype","expected"]].head().to_string())

['prompt_id', 'category', 'status', 'replacement_score', 'total_emb_mass', 'total_err_mass', 'direct_mass', 'direct_share', 'mediated_share', 'emb_mass_by_step', 'err_mass_by_step', 'direct_from_final_token', 'n_steps', 'n_tok', 'n_features', 'mediated_replacement', 'subtype', 'expected']
  prompt_id        subtype expected
0     p0000  capital_short    Paris
1     p0015  capital_short  Nairobi
2     p0022    capital_med     Lima
3     p0023   capital_long     Lima
4     p0026   capital_long  Bangkok


In [35]:
fr["frame"] = np.where(fr.subtype.str.contains("long"), "long",
               np.where(fr.subtype.str.contains("med"), "med", "short"))

piv = fr.pivot_table(index="expected", columns="frame", values="replacement_score", aggfunc="mean")
print(piv.to_string())
print("\ncolumns present:", list(piv.columns))
print("items with both short and long:",
      piv[["short", "long"]].dropna().shape[0] if {"short", "long"} <= set(piv.columns) else 0)

frame            long       med     short
expected                                 
Albany       0.717733       NaN  0.710832
Athens       0.773011  0.741803  0.717021
Austin       0.729508       NaN  0.717264
Bangkok      0.766669  0.735640  0.712418
Brasilia     0.771825  0.737896  0.704371
Cairo        0.767234  0.737726  0.706909
French       0.764479       NaN  0.745001
German       0.753625       NaN  0.741922
Harrisburg   0.723567       NaN  0.717866
Havana       0.769599  0.734126  0.716067
Italian      0.756089       NaN  0.738818
Japanese     0.753887       NaN  0.734438
Lansing      0.710615       NaN  0.700912
Lima         0.771365  0.739685  0.724619
Lincoln      0.728229       NaN  0.710752
Lisbon       0.763116  0.732962  0.701689
Madison      0.730911       NaN  0.717798
Nairobi      0.766631  0.732227  0.726235
Nashville    0.730612       NaN  0.719996
Olympia      0.703997       NaN  0.697142
Oslo         0.765129  0.732085  0.710679
Paris        0.777850  0.745978  0

In [36]:
piv = fr.pivot_table(index="expected", columns="frame", values="replacement_score", aggfunc="mean")
print(piv.to_string())
print("\nitems with both short and long:", piv[["short","long"]].dropna().shape[0]
      if {"short","long"} <= set(piv.columns) else "a frame column is missing entirely")

frame            long       med     short
expected                                 
Albany       0.717733       NaN  0.710832
Athens       0.773011  0.741803  0.717021
Austin       0.729508       NaN  0.717264
Bangkok      0.766669  0.735640  0.712418
Brasilia     0.771825  0.737896  0.704371
Cairo        0.767234  0.737726  0.706909
French       0.764479       NaN  0.745001
German       0.753625       NaN  0.741922
Harrisburg   0.723567       NaN  0.717866
Havana       0.769599  0.734126  0.716067
Italian      0.756089       NaN  0.738818
Japanese     0.753887       NaN  0.734438
Lansing      0.710615       NaN  0.700912
Lima         0.771365  0.739685  0.724619
Lincoln      0.728229       NaN  0.710752
Lisbon       0.763116  0.732962  0.701689
Madison      0.730911       NaN  0.717798
Nairobi      0.766631  0.732227  0.726235
Nashville    0.730612       NaN  0.719996
Olympia      0.703997       NaN  0.697142
Oslo         0.765129  0.732085  0.710679
Paris        0.777850  0.745978  0

In [37]:
# 3. does the category ranking survive removing the direct component?
rank = pd.DataFrame({
    "replacement": dp.groupby("category").replacement_score.mean(),
    "mediated": dp.groupby("category").mediated_replacement.mean(),
    "direct_share": dp.groupby("category").direct_share.mean()}).round(4)
rank["rank_replacement"] = rank.replacement.rank(ascending=False).astype(int)
rank["rank_mediated"] = rank.mediated.rank(ascending=False).astype(int)
print(rank.sort_values("rank_replacement").to_string())
rho, p = stats.spearmanr(rank.rank_replacement, rank.rank_mediated)
print(f"\nSpearman(replacement rank, mediated rank) = {rho:+.3f} (p = {p:.3f})")

PREDICTED = ["factual_recall", "syntactic_agreement", "arithmetic", "entity_known_unknown",
             "multi_hop", "code_completion", "induction", "obfuscated"]
obs_med = rank.sort_values("mediated", ascending=False).index.tolist()
pr = {c: i for i, c in enumerate(PREDICTED)}; ob = {c: i for i, c in enumerate(obs_med)}
r1, p1 = stats.spearmanr([pr[c] for c in PREDICTED], [ob[c] for c in PREDICTED])
print(f"P1 re-scored on mediated replacement: Spearman = {r1:+.3f} (p = {p1:.3f})")
print("(P1 on raw replacement was +0.071, p = 0.867 — this is exploratory, not a re-test)")

                      replacement  mediated  direct_share  rank_replacement  rank_mediated
category                                                                                  
induction                  0.7455    0.6509        0.1270                 1              6
entity_known_unknown       0.7424    0.6758        0.0897                 2              3
syntactic_agreement        0.7415    0.6745        0.0900                 3              4
factual_recall             0.7381    0.6470        0.1235                 4              7
code_completion            0.7357    0.6787        0.0770                 5              1
multi_hop                  0.7279    0.6580        0.0962                 6              5
obfuscated                 0.7273    0.6761        0.0704                 7              2
arithmetic                 0.7091    0.6434        0.0927                 8              8

Spearman(replacement rank, mediated rank) = +0.071 (p = 0.867)
P1 re-scored on mediated r

## Part 7 — Save

In [38]:
dp.to_csv(f"{OUT}/direct_path.csv", index=False)
summary = dict(
    n_graphs=int(len(dp)),
    direct_share=dict(mean=float(dp.direct_share.mean()), median=float(dp.direct_share.median()),
                      by_category=dp.groupby("category").direct_share.mean().round(4).to_dict()),
    corr_direct_replacement=float(dp.direct_share.corr(dp.replacement_score)),
    eta2=dict(replacement=eta2(dp.replacement_score, cat),
              direct_share=eta2(dp.direct_share, cat),
              mediated=eta2(dp.mediated_replacement, cat)),
    day6_parsed_estimate=dict(direct_share=0.372, corr_replacement=0.556, corr_rating=0.033),
)
json.dump(summary, open(f"{OUT}/day7_direct_path.json", "w"), indent=2, default=float)
print(json.dumps(summary, indent=2)[:1200])
print("\n*** DOWNLOAD /kaggle/working/out/ ***")

{
  "n_graphs": 255,
  "direct_share": {
    "mean": 0.09492521332295101,
    "median": 0.09139372075138327,
    "by_category": {
      "arithmetic": 0.0927,
      "code_completion": 0.077,
      "entity_known_unknown": 0.0897,
      "factual_recall": 0.1235,
      "induction": 0.127,
      "multi_hop": 0.0962,
      "obfuscated": 0.0704,
      "syntactic_agreement": 0.09
    }
  },
  "corr_direct_replacement": 0.15738070462981396,
  "eta2": {
    "replacement": 0.24229245615262487,
    "direct_share": 0.46715393480211653,
    "mediated": 0.2842785742718085
  },
  "day6_parsed_estimate": {
    "direct_share": 0.372,
    "corr_replacement": 0.556,
    "corr_rating": 0.033
  }
}

*** DOWNLOAD /kaggle/working/out/ ***


## Record in the pre-registration

> **Direct token-to-logit share, computed exactly [DATE].** The Day 6 figure was parsed from the
> displayed trace edges on the 40 rated graphs. Recomputed from the full influence vector on [__]
> graphs with no thresholding: mean direct share [__] (parsed estimate 0.372), correlation with
> replacement score [__] (parsed +0.556), with human rating [__] (parsed +0.033). Replacement
> scores recomputed here match the corpus run to [__], within the re-attribution ceiling.
>
> Exploratory: splitting replacement into direct and mediated components gives category eta² of
> [__] and [__] respectively, against [__] for replacement as a whole. The matched short/long frame
> effect on [replacement / direct share / mediated replacement] is [__].